In [1]:
import pandas as pd
import pycountry

alm = pd.read_csv('../data/alarm_clean.csv', encoding='utf-8-sig')
sft = pd.read_csv('../data/safety_clean.csv', encoding='utf-8-sig')
ntc = pd.read_csv('../data/safety_notice_processed.csv', encoding='utf-8-sig')
inc = pd.read_csv('../data/incident_info_clean.csv', encoding='utf-8-sig')
dst = pd.read_csv('../data/destinations_clean.csv', encoding='utf-8-sig')


### ISO코드(2글자)만 있는 파일에 iso3 추가

In [2]:
def to_iso3(iso2):
    c = pycountry.countries.get(alpha_2=str(iso2).strip().upper())
    return c.alpha_3 if c else None

ntc['iso3'] = ntc['ISO코드'].apply(to_iso3)
inc['iso3'] = inc['ISO코드'].apply(to_iso3)

files = {'여행지': dst, '여행경보': alm, '안전정보': sft, '안전공지': ntc, '사건사고': inc}
for name, df in files.items():
    print(f"{name:6} {len(df):>5}행  {df['iso3'].nunique():>3}개국  iso3결측 {df['iso3'].isna().sum()}")

여행지      111행   43개국  iso3결측 2
여행경보     208행  145개국  iso3결측 2
안전정보    5196행  153개국  iso3결측 271
안전공지    4999행  177개국  iso3결측 39
사건사고     198행  196개국  iso3결측 2


### 어떤 나라 빠지는지 확인

In [3]:
for name, df in files.items():
    miss = df[df['iso3'].isna()]['국가명'].unique() if '국가명' in df.columns else df[df['iso3'].isna()]['country_en'].unique()
    print(f"{name}: {list(miss)[:10]}")

여행지: ['South Korea']
여행경보: ['코소보']
안전정보: ['러시아', '홍콩', '콩고민주공화국', '코트디부아르', '브루나이', '마카오', '코스타리카', '솔로몬제도', '동티모르', '카보베르데']
안전공지: ['ALL', '나미비아']
사건사고: ['나미비아', '코소보']


In [4]:
ds = set(dst['iso3'].dropna())

for name, df in files.items():
    if name == '여행지':
        continue
    print(f"여행지 ∩ {name}: {len(ds & set(df['iso3'].dropna()))}개국 / {len(ds)}")

여행지 ∩ 여행경보: 31개국 / 43
여행지 ∩ 안전정보: 43개국 / 43
여행지 ∩ 안전공지: 43개국 / 43
여행지 ∩ 사건사고: 43개국 / 43
